# 3 — 声を使う

読み上げたい日本語テキストを入力し、生成音声を WAV で保存します。

**前提:** 「声を覚える」と同じ Colab ランタイムを使い、GPT-SoVITS のセットアップが完了していること。

In [ ]:
import os, sys, subprocess, pathlib, time
import requests
from IPython.display import Audio, display

WORK = pathlib.Path('/content')
REPO = WORK / 'GPT-SoVITS'
OUT = WORK / 'voice_clone_output'
OUT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    raise FileNotFoundError('先に 2 — 声を覚える Notebook を同じランタイムで実行してください。')


## 1. 参照音声と読み上げ文を設定

下のパスは「声を録る」Notebook の出力に合わせてください。

In [ ]:
# 必要に応じて、録音 Notebook の出力先に合わせて変更
AUDIO_INPUT = pathlib.Path('/content/voice_clone_data/voice_mono_32k.wav')
TRANSCRIPT_INPUT = pathlib.Path('/content/voice_clone_data/reference_transcript.txt')
if not AUDIO_INPUT.is_file():
    raise FileNotFoundError(f'参照音声がありません: {AUDIO_INPUT}')
if not TRANSCRIPT_INPUT.is_file():
    raise FileNotFoundError(f'文字起こしがありません: {TRANSCRIPT_INPUT}')
prompt_text = TRANSCRIPT_INPUT.read_text(encoding='utf-8').strip()
TARGET_TEXT = 'こんにちは。これは、自分の声を使った日本語音声合成のテストです。'
print('参照音声:', AUDIO_INPUT)
print('生成する文:', TARGET_TEXT)
display(Audio(filename=str(AUDIO_INPUT)))

## 2. 推論 API を起動

API の起動に時間がかかる場合があります。

In [ ]:
os.chdir(REPO)
api_log_path = OUT / 'api.log'
api_log = api_log_path.open('w')
api_process = subprocess.Popen(
    [sys.executable, 'api_v2.py', '-a', '127.0.0.1', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
    cwd=str(REPO), stdout=api_log, stderr=subprocess.STDOUT
)
API_URL = 'http://127.0.0.1:9880'
ready = False
for _ in range(180):
    if api_process.poll() is not None:
        break
    try:
        requests.get(API_URL, timeout=2)
        ready = True
        break
    except requests.RequestException:
        time.sleep(2)
if not ready:
    api_log.flush()
    print('\\n'.join(api_log_path.read_text(errors='replace').splitlines()[-100:]))
    raise RuntimeError('API が起動しませんでした。ログを確認してください。')
print('API 起動済み:', API_URL)

## 3. 共通推論スクリプトで生成

`inference.py` を取得して利用し、生成した WAV を再生します。

In [ ]:
INFERENCE_URL = 'https://raw.githubusercontent.com/bonsai/kaggle-jp-tts-guide/master/inference.py'
INFERENCE_SCRIPT = WORK / 'inference.py'
script_response = requests.get(INFERENCE_URL, timeout=30)
script_response.raise_for_status()
INFERENCE_SCRIPT.write_text(script_response.text, encoding='utf-8')
OUTPUT_WAV = OUT / 'generated.wav'
result = subprocess.run([
    sys.executable, str(INFERENCE_SCRIPT),
    '--text', TARGET_TEXT,
    '--ref-audio', str(AUDIO_INPUT),
    '--prompt-text', prompt_text,
    '--output', str(OUTPUT_WAV),
    '--api-url', API_URL,
], text=True, capture_output=True, timeout=660)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'inference.py failed: {result.returncode}')
print('保存先:', OUTPUT_WAV)
display(Audio(filename=str(OUTPUT_WAV)))

## 完了

`/content/voice_clone_output/generated.wav` を保存してください。Colab のランタイム終了後にファイルが消える場合があります。